In [1]:
import torch
from torchvision import transforms
from PIL import Image
import numpy as np

In [9]:
# Define the model (same as during training)
class HybridModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.AdaptiveAvgPool2d((8, 8)))
        self.transformer = torch.nn.Sequential(
            torch.nn.Flatten(start_dim=1),
            torch.nn.Linear(128*8*8, 512),
            torch.nn.Unflatten(1, (64, 8)),
            torch.nn.TransformerEncoder(
                torch.nn.TransformerEncoderLayer(
                    d_model=8, nhead=4, dim_feedforward=32, dropout=0.1, batch_first=True),
                num_layers=1),
            torch.nn.Flatten())
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(64*8, 128),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(128, 3))  # NUM_CLASSES=3

    def forward(self, x):
        x = self.cnn(x)
        x = self.transformer(x)
        return self.classifier(x)
# Initialize model and load weights
model = HybridModel()
model.load_state_dict(torch.load('best_model.pth', map_location=torch.device('cpu')))  # Load to CPU
model.eval()  # Set to evaluation mode

HybridModel(
  (cnn): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): AdaptiveAvgPool2d(output_size=(8, 8))
  )
  (transformer): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=8192, out_features=512, bias=True)
    (2): Unflatten(dim=1, unflattened_size=(64, 8))
    (3): TransformerEncoder(
      (layers): ModuleList(
        (0): TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=8, out_features=8, bias=True)
          )
          (linear1): Linear(in_features=

In [26]:
# Define transforms (must match test_transform during training)
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

def preprocess_image(image_path):
    img = Image.open(image_path).convert('L')  # Convert to grayscale
    img = transform(img)
    img = img.unsqueeze(0)  # Add batch dimension (B=1)
    return img

# Example usage
image_path = 'split_data/test/pneumonia/person1034_virus_1728.jpeg'
input_tensor = preprocess_image(image_path)

In [27]:
# Predict
with torch.no_grad():
    outputs = model(input_tensor)
    probabilities = torch.nn.functional.softmax(outputs, dim=1)
    predicted_class = torch.argmax(probabilities, dim=1).item()

# Map class index to label
class_names = ['covid', 'normal', 'pneumonia']
predicted_label = class_names[predicted_class]

print(f"Predicted class: {predicted_label}")
print(f"Class probabilities: {probabilities.squeeze().numpy()}")

Predicted class: pneumonia
Class probabilities: [5.5080052e-02 1.8114115e-04 9.4473881e-01]
